In [1]:

# Import pandas for data processing.
import pandas as pd

# Import Path for managing file paths.
from pathlib import Path

# Define the path to the raw test dataset.
raw_file = Path(
    "../../Raw_Data/Disaster_Response_management_Data/"
    "disaster_response_test.csv"
)

# Define the output folder for processed data and reports.
output_folder = Path(".")

# Create the output folder if it does not exist.
output_folder.mkdir(parents=True, exist_ok=True)

# Load only the test dataset.
df = pd.read_csv(raw_file, low_memory=False)

# Record the original dataset dimensions.
original_rows = len(df)
original_columns = len(df.columns)

# Count missing values before preprocessing.
missing_before = int(df.isna().sum().sum())

# Remove columns that are completely empty.
empty_columns = df.columns[df.isna().all()].tolist()
df = df.drop(columns=empty_columns)

# Remove leading and trailing spaces from text fields.
# Preserve missing values instead of converting them to strings.
text_columns = ["message", "original", "genre", "split", "event"]

for column in text_columns:
    if column in df.columns:
        df[column] = df[column].map(
            lambda value: value.strip()
            if isinstance(value, str)
            else value
        )

# Convert the Haiti date column to datetime format.
# Invalid dates become missing values.
if "date_haiti" in df.columns:
    df["date_haiti"] = pd.to_datetime(
        df["date_haiti"],
        errors="coerce"
    )

# Count exact duplicate rows before removing them.
duplicate_rows = int(df.duplicated().sum())

# Remove exact duplicate rows, keeping the first occurrence.
df = df.drop_duplicates().reset_index(drop=True)

# Count repeated IDs after duplicate-row removal.
duplicate_ids = (
    int(df["id"].duplicated().sum())
    if "id" in df.columns else 0
)

# Count missing message text.
missing_messages = (
    int(df["message"].isna().sum())
    if "message" in df.columns else 0
)

# Count missing values after preprocessing.
missing_after = int(df.isna().sum().sum())

# Save the processed test dataset separately.
output_file = output_folder / "disaster_response_test_processed.csv"
df.to_csv(output_file, index=False)

# Build a report containing the test dataset's preprocessing results.
report = pd.DataFrame([{
    "Original_Rows": original_rows,
    "Processed_Rows": len(df),
    "Original_Columns": original_columns,
    "Processed_Columns": len(df.columns),
    "Empty_Columns_Removed": (
        ", ".join(empty_columns) if empty_columns else "None"
    ),
    "Duplicate_Rows_Removed": duplicate_rows,
    "Duplicate_ID_Count": duplicate_ids,
    "Missing_Messages": missing_messages,
    "Missing_Values_Before": missing_before,
    "Missing_Values_After": missing_after
}])

# Save the test preprocessing report separately.
report_file = output_folder / "disaster_response_test_preprocessing_report.csv"
report.to_csv(report_file, index=False)

# Print the preprocessing results.
print("--- TEST DATA PREPROCESSING RESULTS ---")
print("Original shape:", (original_rows, original_columns))
print("Processed shape:", df.shape)
print("Empty columns removed:", empty_columns)
print("Duplicate rows removed:", duplicate_rows)
print("Duplicate IDs remaining:", duplicate_ids)
print("Missing messages:", missing_messages)
print("Missing values before:", missing_before)
print("Missing values after:", missing_after)
print("\nProcessed CSV saved to:", output_file.resolve())
print("Preprocessing report saved to:", report_file.resolve())

# Display the report as a table.
display(report)


--- TEST DATA PREPROCESSING RESULTS ---
Original shape: (2397, 45)
Processed shape: (2387, 45)
Empty columns removed: []
Duplicate rows removed: 10
Duplicate IDs remaining: 0
Missing messages: 0
Missing values before: 5551
Missing values after: 5513

Processed CSV saved to: C:\Users\ishit\jeevan_link_ml\Data\Processed_data\Disaster Response Messages\disaster_response_test_processed.csv
Preprocessing report saved to: C:\Users\ishit\jeevan_link_ml\Data\Processed_data\Disaster Response Messages\disaster_response_test_preprocessing_report.csv


,Original_Rows,Processed_Rows,Original_Columns,Processed_Columns,Empty_Columns_Removed,Duplicate_Rows_Removed,Duplicate_ID_Count,Missing_Messages,Missing_Values_Before,Missing_Values_After
0,2397,2387,45,45,None,10,0,0,5551,5513
